# Load a PDF

In [1]:
from pypdf import PdfReader

pdf_path = r"C:\Users\Pratik\Desktop\rag\RAG\data\git-cheat-sheet-education.pdf"

reader = PdfReader(pdf_path)

text = ""

for page_number, page in enumerate(reader.pages):
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print(text[:3000])

Ignoring wrong pointing object 11 0 (offset 0)


GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show modiﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now to your next commit (stage)
git reset [file]
unstage a ﬁle while retaining the changes in working directory
git diff
diﬀ of what is changed but not staged
git diff --staged
diﬀ of what is staged but not yet committed
git commit -m “[descriptive message]”
commit your staged content as a new commit snapshot
SETUP
Conﬁguring user information used across all local repositories
git config --global user.name “[firstname lastname]”
set a name that is identiﬁable for credit when review version history
git config --global user.email “[valid-email]”
set an email address that will be associated with each history marker
git config --global color.ui auto
set automatic command line coloring for Git for easy reviewing
SETUP & INIT
Conﬁguring user information, initializing and cloning repositories
git in

# Process Page-by-Page
documents contain multiple pages.

We can also preserve metadata.

Instead of:

text = reader.pages[0].extract_text()

we process:

for page_number, page in enumerate(reader.pages):

# Text Cleaning

Raw extracted PDF text can be messy.

For example:

Employees are entitled to

20 days of annual
leave per year.

Then:

cleaned_text = clean_text(page_text)

Now:

Employees are entitled to 20 days of annual leave per year.


``` Cleaning should normalize noise without destroying semantic information. ```

In [4]:
import re

def clean_text(text):

    #removing excessiven whitespaces
    text = re.sub(r"\s"," ",text)

    #removing leading, trailing whitespaces
    text = text.strip()

    return text

clean_text(text)


"GIT CHEAT SHEET STAGE & SNAPSHOT Working with snapshots and the Git staging area git status show modiﬁed ﬁles in working directory, staged for your next commit git add [file] add a ﬁle as it looks now to your next commit (stage) git reset [file] unstage a ﬁle while retaining the changes in working directory git diff diﬀ of what is changed but not staged git diff --staged diﬀ of what is staged but not yet committed git commit -m “[descriptive message]” commit your staged content as a new commit snapshot SETUP Conﬁguring user information used across all local repositories git config --global user.name “[firstname lastname]” set a name that is identiﬁable for credit when review version history git config --global user.email “[valid-email]” set an email address that will be associated with each history marker git config --global color.ui auto set automatic command line coloring for Git for easy reviewing SETUP & INIT Conﬁguring user information, initializing and cloning repositories git i

# Chunking

Suppose our document contains:

The company provides employees with medical insurance.

Employees become eligible after completing 90 days.

The insurance covers hospitalization expenses.

Employees must submit claims within 30 days...

We shouldn't necessarily embed the entire document as one vector.

Instead:

Document

   │

   ├── Chunk 1

   ├── Chunk 2

   ├── Chunk 3
   
   └── Chunk 4

Why?

Because retrieval works at the chunk level.

# Naive Chunking
The simplest approach

In [5]:
def chunk_text(text,chunk_size=500):
    chunks = []

    for i in range(0,len(text),chunk_size):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)

    return chunks

In [8]:
#This is character-based chunking.

chunks = chunk_text(text,100)

for chunk in chunks:
    print("------")
    print(chunk)

------
GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show mod
------
iﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now
------
 to your next commit (stage)
git reset [file]
unstage a ﬁle while retaining the changes in working d
------
irectory
git diff
diﬀ of what is changed but not staged
git diff --staged
diﬀ of what is staged but 
------
not yet committed
git commit -m “[descriptive message]”
commit your staged content as a new commit s
------
napshot
SETUP
Conﬁguring user information used across all local repositories
git config --global use
------
r.name “[firstname lastname]”
set a name that is identiﬁable for credit when review version history

------
git config --global user.email “[valid-email]”
set an email address that will be associated with eac
------
h history marker
git config --global color.ui auto
set automatic command line coloring for Git for e
------
asy reviewing
SETUP &

but this can have a problem:

### `` The Chunk Boundary Problem``

for example:

Imagine:

``Chunk 1``

Employees receive 20 days
of annual leave. Employees
can carry forward up to...

and:

`Chunk 2`

10 unused days into the
following year.

The information:

20 days
+
carry forward
+
10 days

has been split.

A query like:

How many leave days can employees carry forward?

may retrieve only Chunk 2.

The answer is incomplete.

This is why chunking is not just splitting text.


One of the solution to this is chunk Overlap
Instead of:

Chunk 1
[A B C D E]

Chunk 2
[F G H I J]

we do:

Chunk 1
[A B C D E]

Chunk 2
[D E F G H]

Chunk 3
[G H I J K]

The overlapping information helps preserve context.

In [10]:
def chunk_text_overlap(text,chunk_size=500,overlap =100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [11]:
chunks = chunk_text_overlap(text,chunk_size = 500)

for i, chunk in enumerate(chunks):
    print(f"\n -- chunk{i}---")
    print(chunk)


 -- chunk0---
GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show modiﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now to your next commit (stage)
git reset [file]
unstage a ﬁle while retaining the changes in working directory
git diff
diﬀ of what is changed but not staged
git diff --staged
diﬀ of what is staged but not yet committed
git commit -m “[descriptive message]”
commit your staged content as a new commit s

 -- chunk1---
not yet committed
git commit -m “[descriptive message]”
commit your staged content as a new commit snapshot
SETUP
Conﬁguring user information used across all local repositories
git config --global user.name “[firstname lastname]”
set a name that is identiﬁable for credit when review version history
git config --global user.email “[valid-email]”
set an email address that will be associated with each history marker
git config --global color.ui auto
set automatic comm

The important calculation is:

``start += chunk_size - overlap``

For:

chunk_size = 500
overlap = 100

we move:

500 - 100 = 400

characters each time.

But Character Chunking Isn't Ideal

Suppose we have:

Machine learning is a field of artificial intelligence.

Deep learning is a subset of machine learning.

Transformers are neural network architectures...

Character splitting doesn't understand:

sentences
paragraphs
sections
topics

It simply counts characters.

A better approach is `` recursive chunking.``

## Recursive Chunking

The idea is:

Try paragraph
      ->
If too large
    ->
Try sentence
      ->
If still too large
      ->
Try words
      ->
If still too large
      ->
Split characters

Conceptually:

Document

   │
   ├── Paragraph

   │      │

   │                     ├── Sentence
   │      │

   │                     └── Sentence
   │

   └── Paragraph

This usually produces much more meaningful chunks.

# Using LangChain's Recursive Splitter

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap =100)

chunks = splitter.split_text(text)

for i,chunk in enumerate(chunks):
    print(f"\n---chunk {i}---")
    print(chunk)


---chunk 0---
GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show modiﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now to your next commit (stage)
git reset [file]
unstage a ﬁle while retaining the changes in working directory
git diff
diﬀ of what is changed but not staged
git diff --staged
diﬀ of what is staged but not yet committed
git commit -m “[descriptive message]”

---chunk 1---
git diff --staged
diﬀ of what is staged but not yet committed
git commit -m “[descriptive message]”
commit your staged content as a new commit snapshot
SETUP
Conﬁguring user information used across all local repositories
git config --global user.name “[firstname lastname]”
set a name that is identiﬁable for credit when review version history
git config --global user.email “[valid-email]”
set an email address that will be associated with each history marker
git config --global color.ui auto

---chunk 2---
git 

In [3]:
#ingestion pipeline

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

pdf = r"C:\Users\Pratik\Desktop\rag\RAG\data\git-cheat-sheet-education.pdf"

#load the pdf 

reader = PdfReader(pdf)

pages = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if not text:
        continue

    text = re.sub(r"\s+"," ",text).strip()

    pages.append({
        "text": text,
        "page" : page_number + 1,
        "source" : pdf
    })


    #chunk documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 100
    )

    chunks = []

    for page in pages:

        page_chunks = splitter.split_text(page["text"])

        for chunk_number, chunk_text in enumerate(page_chunks):

            chunks.append({
                "text" : chunk_text,
                "metadata" : page["source"],
                "page" : page["page"],
                "chunk_id" : chunk_number
            })

#inscpect chunks

print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks[:5]):

    print("\n +++++++++++++++")

    print("chunk: ",i)

    print("Metadata: ")
    print(chunk["metadata"])

    print("\n text")
    print(chunk["text"])

Ignoring wrong pointing object 11 0 (offset 0)


Total chunks: 12

 +++++++++++++++
chunk:  0
Metadata: 
C:\Users\Pratik\Desktop\rag\RAG\data\git-cheat-sheet-education.pdf

 text
GIT CHEAT SHEET STAGE & SNAPSHOT Working with snapshots and the Git staging area git status show modiﬁed ﬁles in working directory, staged for your next commit git add [file] add a ﬁle as it looks now to your next commit (stage) git reset [file] unstage a ﬁle while retaining the changes in working directory git diff diﬀ of what is changed but not staged git diff --staged diﬀ of what is staged but not yet committed git commit -m “[descriptive message]” commit your staged content as a new commit

 +++++++++++++++
chunk:  1
Metadata: 
C:\Users\Pratik\Desktop\rag\RAG\data\git-cheat-sheet-education.pdf

 text
not yet committed git commit -m “[descriptive message]” commit your staged content as a new commit snapshot SETUP Conﬁguring user information used across all local repositories git config --global user.name “[firstname lastname]” set a name that is identiﬁab

# Entire flow that will be build


                    INGESTION
                        │
                        ▼
                  ┌───────────┐
                  │ Documents │
                  └─────┬─────┘
                        │
                        ▼
                  Text Extraction
                        │
                        ▼
                     Chunking
                        │
                        ▼
                   Embeddings
                        │
                        ▼
                 Vector Database
                        │
        ────────────────┼────────────────
                        │
                        ▼
                    RETRIEVAL
                        │
                  User Question
                        │
                        ▼
                  Query Embedding
                        │
                        ▼
                 Similarity Search
                        │
                        ▼
                  Top-K Chunks
                        │
                        ▼
                    GENERATION
                        │
                        ▼
                Context + Question
                        │
                        ▼
                       LLM
                        │
                        ▼
                     Answer